In [1]:
instruction_data_path = "../data/deposit-account-agreement.jsonl"
from datasets import load_dataset

/Users/nkoneru/vscode/WorkingSamples/Project-one/Project-one/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)
print(instruction_dataset)
print(instruction_dataset[0])

Dataset({
    features: ['prompt', 'question', 'answer'],
    num_rows: 743
})
{'prompt': 'You are a helpful assistant who can answer questions about the topic in the dataset.', 'question': 'What is the effective date of my deposit account agreement with JPMorgan Chase Bank, N.A.?', 'answer': '6/14/2026'}


In [3]:
# ============================================================
# Format instruction records
# ============================================================
# We convert every record into Alpaca-style training text.

def format_instruction_record(record):
    instruction = str(record.get("Prompt", "")).strip()
    input_text = str(record.get("Question", "")).strip()
    output_text = str(record.get("Answer", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [4]:
instruction_dataset = instruction_dataset.map(format_instruction_record)


In [5]:
instruction_dataset


Dataset({
    features: ['prompt', 'question', 'answer', 'text'],
    num_rows: 743
})

In [6]:
print(instruction_dataset[5]["text"])

### Instruction:


### Response:



In [7]:
# ============================================================
# Create train-validation split
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['prompt', 'question', 'answer', 'text'],
        num_rows: 631
    })
    validation: Dataset({
        features: ['prompt', 'question', 'answer', 'text'],
        num_rows: 112
    })
})
Train examples: 631
Validation examples: 112


In [8]:
#!uv pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes torchao

In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "../source/deposit-account-agreement.pdf"

    # Base causal language model that we will fine-tune on customer account-agreement text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "../output/daa_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "../adapter/daa_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "../output/daa_tinyllama_lora_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 10.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

In [11]:
config = Config()

In [12]:
config

Config(pdf_path='../source/deposit-account-agreement.pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='../output/daa_tinyllama_lora_output', adapter_dir='../adapter/daa_tinyllama_lora_adapter', processed_data_dir='../output/daa_tinyllama_lora_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=10.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [13]:
import json
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "../source/deposit-account-agreement.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "../output/daa_tinyllama_lora_output",
  "adapter_dir": "../adapter/daa_tinyllama_lora_adapter",
  "processed_data_dir": "../output/daa_tinyllama_lora_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 10.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [14]:
import os
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [15]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

pdf_pages = extract_pdf_pages(config.pdf_path)
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")



PDF found: ../source/deposit-account-agreement.pdf
Total pages with extracted text: 32
Page-level character counts:
Page 1: 2261 characters
Page 2: 8084 characters
Page 3: 7870 characters
Page 4: 5387 characters
Page 5: 6040 characters
Page 6: 7635 characters
Page 7: 8243 characters
Page 8: 6866 characters
Page 9: 6677 characters
Page 10: 6768 characters
Page 11: 5737 characters
Page 12: 6392 characters
Page 13: 5616 characters
Page 14: 6890 characters
Page 15: 5750 characters
Page 16: 7950 characters
Page 17: 6653 characters
Page 18: 4927 characters
Page 19: 5498 characters
Page 20: 6152 characters
Page 21: 6266 characters
Page 22: 7516 characters
Page 23: 6162 characters
Page 24: 6096 characters
Page 25: 7421 characters
Page 26: 8029 characters
Page 27: 8249 characters
Page 28: 6426 characters
Page 29: 8020 characters
Page 30: 6655 characters
Page 31: 2362 characters
Page 32: 3674 characters


In [16]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

cleaned_pages = []

In [17]:
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 32


In [18]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# This step converts cleaned page-level text into paragraph-level records.

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [19]:
paragraph_records = split_into_paragraph_records(cleaned_pages)
print("Total paragraph records:", len(paragraph_records))

Total paragraph records: 38


In [20]:
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

Page: 1 | Paragraph: 1 | Characters: 2221
DEPOSIT ACCOUNT AGREEMENT JPMorgan Chase Bank, N.A. Member FDIC © 2026 JPMorgan Chase & Co. Page 1 of 30 Effective 6/14/2026 DEPOSIT ACCOUNT AGREEMENT AND PRIVACY NOTICE Thank you for choosing Chase This is your Deposit Account Agreement, or contract, with us. This agreement applies to all Chase personal and business and J.P. Morgan Private Client deposit accounts and the terms and conditions are identical for all of these accounts. We recommend keeping this agreement but we regularly update it, so you can always get the current agreement at chase.com, a branch or by request when you call us. The Deposit Account Agreement also includes these separate documents that pertain to our personal and business accounts: • Rates for interest-bearing accounts • Personal accounts: -Additional Banking Services and Fees for Personal Accounts -J.P. Morgan Private Client Additional Banking Services and Fees • Business accounts: Additional Banking Services and 

In [21]:

# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In real projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "daa_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: ../output/daa_tinyllama_lora_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: ../output/daa_tinyllama_lora_processed_data/daa_paragraph_process.jsonl


In [22]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================
from datasets import Dataset
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )
text_dataset = Dataset.from_list(paragraph_records)

In [23]:
# ============================================================
# 10. Train/eval split
# ============================================================
# Even for small demos, keep an evaluation set.
# This gives us validation loss and perplexity.

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

from datasets import DatasetDict
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 6
    })
})


In [ ]:
import os
from datasets import load_dataset
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
##print("HF_TOKEN loaded from .env:", api_key)

# 1. Load your local or existing dataset
# (Replace this with how you currently load your dataset object)
#dataset = load_dataset("json", data_files="your_data.json") 

# 2. Push directly using your specific username and Write token
# This bypasses all .env caching issues entirely
text_dataset.push_to_hub(
    repo_id="Nashxi/daa-non-instruction-DS",  # Must include username/
    token=api_key       # Paste the raw token string
)

In [25]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
# The tokenizer converts text into token IDs for model training.

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


In [26]:
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


In [27]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [28]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus: 100%|██████████| 6/6 [00:00<00:00, 592.43 examples/s]


In [29]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 6
    })
})

In [30]:
tokenized_datasets['train']['input_ids'][0]

[1,
 9305,
 29871,
 29906,
 310,
 29871,
 29906,
 349,
 10461,
 29871,
 29906,
 11644,
 591,
 526,
 11644,
 338,
 13138,
 445,
 8369,
 29973,
 450,
 501,
 29889,
 29903,
 29889,
 21691,
 18161,
 14582,
 2629,
 278,
 435,
 13427,
 6388,
 1451,
 559,
 3942,
 29892,
 3704,
 435,
 13427,
 6388,
 678,
 559,
 10253,
 29892,
 405,
 29889,
 29909,
 1696,
 678,
 559,
 512,
 7610,
 749,
 29353,
 29892,
 9266,
 1696,
 435,
 29889,
 29925,
 29889,
 20549,
 5356,
 332,
 1907,
 365,
 12182,
 29892,
 450,
 9969,
 8088,
 362,
 9266,
 29889,
 322,
 278,
 25022,
 816,
 3942,
 310,
 14582,
 29892,
 5174,
 988,
 263,
 435,
 13427,
 6388,
 1451,
 559,
 5001,
 5626,
 263,
 5004,
 8369,
 29889,
 1724,
 591,
 437,
 1128,
 947,
 678,
 559,
 12566,
 590,
 7333,
 2472,
 29973,
 1763,
 12566,
 596,
 7333,
 2472,
 515,
 1185,
 329,
 2015,
 1891,
 2130,
 322,
 671,
 29892,
 591,
 671,
 6993,
 15366,
 393,
 752,
 368,
 411,
 17097,
 4307,
 29889,
 4525,
 15366,
 3160,
 6601,
 9437,
 24024,
 3163,
 322,
 26130,
 2066

In [31]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [32]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens: 100%|██████████| 32/32 [00:00<00:00, 1705.48 examples/s]
Creating fixed-size training blocks of 512 tokens: 100%|██████████| 6/6 [00:00<00:00, 1105.75 examples/s]


In [33]:
sample = final_dataset["train"][0]
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Page 2 of 2 PAGE 2 Who we are Who is providing this notice? The U.S. consumer financial companies within the JPMorganChase family, including JPMorgan Chase Bank, N.A., Chase Insurance Agency, Inc., J.P. Morgan Securities LLC, The Infatuation Inc. and the Frosch family of companies, except where a JPMorganChase company issues a separate notice. What we do How does Chase protect my personal information? To protect your personal information from unauthorized access and use, we use security measures that comply with federal law. These measures include computer safeguards and secured files and buildings. We authorize our employees to get your information only when they need it to do their work, and we require companies that work for us to protect your information. How does Chase collect my personal information? We collect your personal information, for example, wh

In [34]:
# ============================================================
# 13. Load base model
# ============================================================
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


In [35]:
# Clear memory before loading the model.
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [36]:
from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

W0724 21:47:27.652000 10996 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0724 21:47:27.685000 10996 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0724 21:47:27.711000 10996 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8008.66it/s]

Base model loaded successfully.


In [37]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.
from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [38]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [39]:
# ============================================================
# 15. Data collator
# ============================================================
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [40]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a small classroom/demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

from transformers import TrainingArguments

In [41]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [42]:
from transformers import TrainingArguments
training_args = TrainingArguments(**training_kwargs)

In [43]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


In [44]:
import warnings
warnings.filterwarnings("ignore")

In [45]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()
print("Training completed.")

Step,Training Loss
1,1.847264
2,1.948926
3,2.086464
4,2.147092
5,1.647085
6,2.068673
7,2.087375
8,1.901032
9,1.911805
10,1.784008


Training completed.


In [46]:
for log in trainer.state.log_history:
    print(log)

{'loss': 1.8472644090652466, 'grad_norm': 0.5554518699645996, 'learning_rate': 0.0, 'epoch': 0.1111111111111111, 'step': 1}
{'loss': 1.9489260911941528, 'grad_norm': 0.4740503430366516, 'learning_rate': 4e-05, 'epoch': 0.2222222222222222, 'step': 2}
{'loss': 2.086463689804077, 'grad_norm': 0.4596669673919678, 'learning_rate': 8e-05, 'epoch': 0.3333333333333333, 'step': 3}
{'loss': 2.1470916271209717, 'grad_norm': 0.5047833323478699, 'learning_rate': 0.00012, 'epoch': 0.4444444444444444, 'step': 4}
{'loss': 1.6470847129821777, 'grad_norm': 0.4566117525100708, 'learning_rate': 0.00016, 'epoch': 0.5555555555555556, 'step': 5}
{'loss': 2.0686728954315186, 'grad_norm': 0.4338705539703369, 'learning_rate': 0.0002, 'epoch': 0.6666666666666666, 'step': 6}
{'loss': 2.0873751640319824, 'grad_norm': 0.4737543761730194, 'learning_rate': 0.00019764705882352942, 'epoch': 0.7777777777777778, 'step': 7}
{'loss': 1.9010318517684937, 'grad_norm': 0.46621838212013245, 'learning_rate': 0.00019529411764705

In [47]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

('../adapter/daa_tinyllama_lora_adapter/tokenizer_config.json',
 '../adapter/daa_tinyllama_lora_adapter/tokenizer.json')

In [48]:
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: ../adapter/daa_tinyllama_lora_adapter
Saved files:
['adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'README.md', 'adapter_config.json']


In [49]:
# ============================================================
# 20. Push LoRA adapter to Hugging Face Hub
# ============================================================
repo_id = "Nashxi/bankaccountagreement-tinyllama-domain-lora-live"

In [50]:
import os
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
#print("HF_TOKEN loaded from .env:", api_key)

model.push_to_hub(
    repo_id, token=api_key,
    private=True
)

Processing Files (1 / 1): 100%|██████████| 50.5MB / 50.5MB, 3.60MB/s  
New Data Upload: 100%|██████████| 50.5MB / 50.5MB, 3.60MB/s  


CommitInfo(commit_url='https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-live/commit/1021557287ea26f272c2ec587fc83533feb49c4d', commit_message='Upload model', commit_description='', oid='1021557287ea26f272c2ec587fc83533feb49c4d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-live', endpoint='https://huggingface.co', repo_type='model', repo_id='Nashxi/bankaccountagreement-tinyllama-domain-lora-live'), pr_revision=None, pr_num=None)

In [51]:
tokenizer.push_to_hub(
    repo_id,
    private=True
)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-live/commit/1021557287ea26f272c2ec587fc83533feb49c4d', commit_message='Upload tokenizer', commit_description='', oid='1021557287ea26f272c2ec587fc83533feb49c4d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-live', endpoint='https://huggingface.co', repo_type='model', repo_id='Nashxi/bankaccountagreement-tinyllama-domain-lora-live'), pr_revision=None, pr_num=None)

In [52]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [53]:
# yo bro - start again with a fresh model and tokenizer load, then apply the LoRA adapter.

from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [54]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5469.41it/s]


In [55]:
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

In [56]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [57]:
print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


In [58]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [59]:
# ============================================================
# 23. Test text continuation
# ============================================================
# These prompts are continuation-style prompts.
# In Notebook 2, we will create instruction prompts for Q&A.

prompts = [
    "Uniform Transfers to Minors Act/Uniform Gifts to Minors Act (UTMA/UGMA) account",
    "When survivorship rights apply Except as otherwise stated in this paragraph",
    "What we do How does Chase protect my personal information?",
    "If a joint account has rights of survivorship, and one joint owner dies",
]

prompts1 = [

    "If one owner of a marital account dies, the survivor is ",
]


In [60]:
#now test this with the generate_completion function
import textwrap
for prompt in prompts1:
    print("=" * 80)
    print(f"Prompt:\n{prompt}\n")
    completion = generate_completion(prompt, max_new_tokens=120)
    print(f"Completion:\n{completion}\n")
    # Wrap and print directly
    print(textwrap.fill(completion, width=80))


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt:
If one owner of a marital account dies, the survivor is 

Completion:
If one owner of a marital account dies, the survivor is 􀂽enrolled in the account and has rights to the funds as if the deceased owner had been a signer on the account during the last known account activity. If there is no surviving owner, we are authorized to pay the funds to the decedent’s estate. If one owner of a non-marital account dies, the survivor is not 􀂽enrolled in the account and has no rights to the funds. If there is no surviving owner, we are authorized to pay the funds to the decedent’s

If one owner of a marital account dies, the survivor is 􀂽enrolled in the account
and has rights to the funds as if the deceased owner had been a signer on the
account during the last known account activity. If there is no surviving owner,
we are authorized to pay the funds to the decedent’s estate. If one owner of a
non-marital account dies, the survivor is not 􀂽enrolled in the account and has
no rights to the f

In [61]:
# note that till now, we have only base model and Lora Adapter
# now i will merge them into a single model

# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

merged_model_dir = "../output/daa_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

offload_model_dir = "../output/daa_tinyllama_offload_model"
os.makedirs(offload_model_dir, exist_ok=True)


In [69]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    offload_folder=offload_model_dir  # Add this line
)


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 150.65it/s]


In [ ]:
# Print the structure to find the true path of the lm_head
print(base_model) 

# Or inspect the state dict keys directly
#print([k for k in base_model.state_dict().keys() if "lm_head" in k])

# Force-feed the script the exact dictionary key string it wants
#state_dict = base_model.state_dict()
#if "lm_head" in state_dict:
#    state_dict["base_model.model.model.lm_head"] = state_dict["lm_head"]

# Or inspect the state dict keys directly
#print([k for k in base_model.state_dict().keys() if "lm_head" in k])

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
import os
import torch
from peft import PeftModel
from safetensors.torch import load_file

sf_path = os.path.join(config.adapter_dir, "adapter_model.safetensors")
bin_path = os.path.join(config.adapter_dir, "adapter_model.bin")

if os.path.exists(sf_path):
    adapter_weights = load_file(sf_path)
else:
    adapter_weights = torch.load(bin_path, map_location="cpu")

# 3. Apply the patch: Remap the flat key to the nested one PEFT wants
#if "lm_head" in adapter_weights:
#    adapter_weights["base_model.model.model.lm_head"] = adapter_weights.pop("lm_head")
#if "lm_head.weight" in adapter_weights:
#    adapter_weights["base_model.model.model.lm_head.weight"] = adapter_weights.pop("lm_head.weight")



In [71]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir,
    low_cpu_mem_usage=False,
    offload_folder=offload_model_dir  # Add this lineoffload_folder=offload_model_dir  # Add this line

)
model_with_adapter.load_state_dict(adapter_weights, strict=False)

_IncompatibleKeys(missing_keys=['base_model.model.model.embed_tokens.weight', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.bas

In [72]:
# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

In [73]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(merged_model_dir)

inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.28s/it]

Merged model saved to: ../output/daa_tinyllama_merged_model


In [ ]:
import os
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# ============================================================
# 20. Push LoRA adapter + base model Merged model to Hugging Face Hub
# ============================================================
repo_id = "Nashxi/bankaccountagreement-tinyllama-domain-lora-merged"

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
#print("HF_TOKEN loaded from .env:", api_key)

merged_model.push_to_hub(
    repo_id, token=api_key,
    private=True
)

In [98]:
# every model push requries a token, so we will push the tokenizer as well
inference_tokenizer.push_to_hub(
    repo_id,
    private=True
)


CommitInfo(commit_url='https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-merged/commit/7b167f2671ea11b447d6cd74fec8ac8935614f8d', commit_message='Upload tokenizer', commit_description='', oid='7b167f2671ea11b447d6cd74fec8ac8935614f8d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='Nashxi/bankaccountagreement-tinyllama-domain-lora-merged'), pr_revision=None, pr_num=None)

# Stage 2: Continue with Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model
 
In Stage 1, we performed **non-instruction fine-tuning / domain-adaptive continued pretraining** on raw DAA PDF text.

Now we continue from the **same Stage 1 LoRA adapter** and perform **instruction fine-tuning** using structured DAA instruction-response examples.

```text
Base TinyLlama
   ↓
Stage 1: Raw DAA text continued pretraining using LoRA
   ↓
Stage 1 domain-adapted LoRA adapter
   ↓
Stage 2: Instruction fine-tuning on deposit agreements Q&A data
   ↓
Final instruction-tuned Deposits Accounts Agreements DAA LoRA adapter
```

This means we are not starting from scratch. We are continuing from the model adapter trained in the previous stage.

What changes in instruction fine-tuning?

For non-instruction fine-tuning, the data looked like raw text:

```text
Any deposit account, such as a checking or savings account, you have with us that is covered by this Agreement....
```

For instruction fine-tuning, the data looks like:

```json
{
  "instruction": "Explain the use of this Agreement.",
  "input": "",
  "output": "Any deposit account, such as a checking or savings account, you have with us that is covered by this Agreement...."
}
```

This teaches the model not only deposit account agreement language, but also how to answer user instructions.

In [75]:
instruction_data_path = "../data/deposit-account-agreement.jsonl"

In [76]:
from datasets import load_dataset
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)

In [77]:
print(instruction_dataset)

Dataset({
    features: ['prompt', 'question', 'answer'],
    num_rows: 743
})


In [78]:
print(instruction_dataset[0])

{'prompt': 'You are a helpful assistant who can answer questions about the topic in the dataset.', 'question': 'What is the effective date of my deposit account agreement with JPMorgan Chase Bank, N.A.?', 'answer': '6/14/2026'}


In [82]:
# ============================================================
# Format instruction records
# ============================================================
# We convert every record into Alpaca-style training text.

def format_instruction_record(record):
    instruction = str(record.get("question", "")).strip()
    input_text = str(record.get("", "")).strip()
    output_text = str(record.get("answer", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [83]:
instruction_dataset = instruction_dataset.map(format_instruction_record)

Map: 100%|██████████| 743/743 [00:00<00:00, 27707.45 examples/s]


In [84]:
print(instruction_dataset[0]["text"])

### Instruction:
What is the effective date of my deposit account agreement with JPMorgan Chase Bank, N.A.?

### Response:
6/14/2026


In [85]:
# ============================================================
# Create train-validation split
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['prompt', 'question', 'answer', 'text'],
        num_rows: 631
    })
    validation: Dataset({
        features: ['prompt', 'question', 'answer', 'text'],
        num_rows: 112
    })
})
Train examples: 631
Validation examples: 112


In [86]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
# The tokenizer converts text into token IDs for model training.

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


In [87]:
instruction_max_length = 512

In [88]:

def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Ignore padding tokens in the loss calculation.
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

In [89]:
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

Tokenizing instruction dataset: 100%|██████████| 112/112 [00:00<00:00, 9481.71 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 631
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 112
    })
})


In [92]:
# ============================================================
# Load merged Stage 1 model and add new LoRA adapter for instruction tuning
# ============================================================

# Merged Stage 1 Model
#    +
# New LoRA adapter for instruction tuning

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

merged_model_dir = "../output/daa_tinyllama_merged_model"

if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False

# Create a new LoRA adapter for instruction fine-tuning.
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4881.84it/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [93]:
# ============================================================
# Instruction fine-tuning data collator
# ============================================================
# This prepares mini-batches for causal language model training.

from transformers import DataCollatorForLanguageModeling
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [94]:
# ============================================================
# Instruction fine-tuning arguments
# ============================================================

instruction_output_dir = "../output/daa_tinyllama_instruction_lora_output"
instruction_adapter_dir = "../output/daa_tinyllama_instruction_lora_adapter"

os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)

In [95]:
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,

    # Train for 5 full epochs.
    num_train_epochs=5,
    max_steps=-1,

    # Batch settings.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings.
    learning_rate=1e-4,
    warmup_steps=5,
    weight_decay=0.01,

    # Show training loss at every step.
    logging_steps=1,
    logging_first_step=True,

    # Run validation at every step.
    eval_strategy="steps",
    eval_steps=1,

    # Save checkpoints.
    save_steps=25,
    save_total_limit=2,

    # Precision settings.
    fp16=use_cuda,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,
)

print(instruction_training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1,
eval_strategy=IntervalStrategy.STEPS,
eval_us

In [96]:
# ============================================================
# Build instruction Trainer
# ============================================================

from transformers import Trainer
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets["train"],
    eval_dataset=instruction_tokenized_datasets["validation"],
    data_collator=instruction_data_collator,
)

print("Instruction Trainer is ready.")

Instruction Trainer is ready.


In [97]:
# ============================================================
# Start instruction fine-tuning
# ============================================================

instruction_train_result = instruction_trainer.train()


Step,Training Loss,Validation Loss
1,3.468283,3.280636
2,3.440938,3.196260
3,3.372315,3.049483
4,3.336797,2.854778
5,3.039235,2.623144
6,2.538703,2.386465
7,2.439598,2.194446
8,2.026451,2.025784
9,2.145161,1.868823
10,1.847132,1.735811


In [99]:
for log in instruction_trainer.state.log_history:
    print(log)

{'loss': 3.468282699584961, 'grad_norm': 6.494777202606201, 'learning_rate': 0.0, 'epoch': 0.012678288431061807, 'step': 1}
{'eval_loss': 3.2806358337402344, 'eval_runtime': 45.4804, 'eval_samples_per_second': 2.463, 'eval_steps_per_second': 2.463, 'epoch': 0.012678288431061807, 'step': 1}
{'loss': 3.4409382343292236, 'grad_norm': 6.95943021774292, 'learning_rate': 2e-05, 'epoch': 0.025356576862123614, 'step': 2}
{'eval_loss': 3.1962597370147705, 'eval_runtime': 45.0246, 'eval_samples_per_second': 2.488, 'eval_steps_per_second': 2.488, 'epoch': 0.025356576862123614, 'step': 2}
{'loss': 3.3723151683807373, 'grad_norm': 6.026582717895508, 'learning_rate': 4e-05, 'epoch': 0.03803486529318542, 'step': 3}
{'eval_loss': 3.049483060836792, 'eval_runtime': 45.2864, 'eval_samples_per_second': 2.473, 'eval_steps_per_second': 2.473, 'epoch': 0.03803486529318542, 'step': 3}
{'loss': 3.336796522140503, 'grad_norm': 4.7857818603515625, 'learning_rate': 6e-05, 'epoch': 0.05071315372424723, 'step': 4}

In [100]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
instruction_model.save_pretrained(instruction_adapter_dir)
tokenizer.save_pretrained(instruction_adapter_dir)

('../output/daa_tinyllama_instruction_lora_adapter/tokenizer_config.json',
 '../output/daa_tinyllama_instruction_lora_adapter/tokenizer.json')

In [101]:
print(f"Instruction LoRA adapter saved to: {instruction_adapter_dir}")
print("Saved files:")
print(os.listdir(instruction_adapter_dir))

Instruction LoRA adapter saved to: ../output/daa_tinyllama_instruction_lora_adapter
Saved files:
['adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'README.md', 'adapter_config.json']


In [104]:
# ============================================================
# 20. Push LoRA adapter to Hugging Face Hub
# ============================================================
repo_id = "Nashxi/bankaccountagreement-tinyllama-domain-lora-instruction"

import os
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
#print("HF_TOKEN loaded from .env:", api_key)

instruction_model.push_to_hub(
    repo_id, token=api_key,
    private=True
)

tokenizer.push_to_hub(
    repo_id,
    private=True
)


Processing Files (1 / 1): 100%|██████████| 50.5MB / 50.5MB, 4.81MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-instruction/commit/13772234d133ef9a229d4e408fdff3a423b901c1', commit_message='Upload tokenizer', commit_description='', oid='13772234d133ef9a229d4e408fdff3a423b901c1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nashxi/bankaccountagreement-tinyllama-domain-lora-instruction', endpoint='https://huggingface.co', repo_type='model', repo_id='Nashxi/bankaccountagreement-tinyllama-domain-lora-instruction'), pr_revision=None, pr_num=None)

In [ ]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================
# Clean old objects to free memory.

del instruction_trainer

try:
    del instruction_model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [105]:
print("Reloading base model + LoRA adapter created a Merged Model. Then, ")
print("the merged model is used wrap another LoRA adapter for instruction fine-tuning")
print("instruction model is uploaded and is complete.")

Reloading base model + LoRA adapter created a Merged Model. Then, 
the merged model is used wrap another LoRA adapter for instruction fine-tuning
instruction model is uploaded and is complete.
